# DeepPCB — Transfer Learning

Detecção dos seis tipos de defeitos com Faster R-CNN ResNet50 FPN V2. O teste oficial permanece separado; treino e validação usam grupos distintos. Escolha A (sem pesos), B (backbone congelado) ou C (fine-tuning completo). O treinamento só começa quando `run_training` for `True`.

## 1. Configuração

A primeira célula reúne os parâmetros em `CONFIG`. A seguinte detecta o ambiente, configura os caminhos e instala no Colab somente pacotes ausentes; depois são carregadas as bibliotecas e selecionado o dispositivo. Localmente, abra o Jupyter pelo ambiente `uv` do projeto.

In [ ]:
# Parâmetros da rodada. Ajuste aqui e reexecute esta célula antes de treinar.
# batch_size e num_workers só valem depois de recriar os DataLoaders (seção 4).
CONFIG = {
    'seed': 42,
    'experiment': 'C',       # A: sem pesos | B: backbone congelado | C: fine-tuning completo
    'epochs': 20,
    'patience': 5,
    'batch_size': 2,
    'num_workers': 0,
    'learning_rate': 0.001,
    'momentum': 0.9,
    'weight_decay': 0.0005,
    'lr_step_size': 7,
    'lr_gamma': 0.1,
    'score_threshold': 0.5,
    'iou_threshold': 0.5,
    'run_training': False,
    'evaluate_test': False,
    'checkpoint_path': None,
}

In [ ]:
# Ambiente e caminhos: não copie o dataset para o runtime do Colab.
import importlib.util
import subprocess
import sys
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except (ImportError, ModuleNotFoundError, ValueError):
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/vision-transfer-learning')
    PCB = ROOT / 'dataset' / 'DeepPCB' / 'PCBData'
    # O Colab já fornece PyTorch/TorchVision; preserve as versões instaladas.
    packages = {
        'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib',
        'PIL': 'pillow', 'tqdm': 'tqdm', 'torch': 'torch',
        'torchvision': 'torchvision', 'torchmetrics': 'torchmetrics',
        'pycocotools': 'pycocotools',
    }
    missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
        importlib.invalidate_caches()
else:
    cwd = Path.cwd().resolve()
    ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'pyproject.toml').is_file()), None)
    if ROOT is None:
        raise RuntimeError('Abra o notebook na pasta do projeto ou em uma subpasta.')
    PCB = ROOT / 'data' / 'DeepPCB' / 'PCBData'
    if Path(sys.prefix).resolve() != (ROOT / '.venv').resolve():
        print('Use o kernel do ambiente uv do projeto: uv run --extra cpu jupyter lab (ou --extra cu128).')

RESULTS = ROOT / 'runs' / 'results'
print('Ambiente:', 'Google Colab' if IN_COLAB else 'local')
print('Dataset:', PCB)
print('Resultados:', RESULTS)

In [ ]:
from pathlib import Path, PurePosixPath
from itertools import combinations
from datetime import datetime
import hashlib, json, math, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
from IPython.display import display
from tqdm.auto import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import pil_to_tensor
from torchvision.ops import box_iou
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision

CLASSES = {1:'open', 2:'short', 3:'mousebite', 4:'spur', 5:'copper', 6:'pin-hole'}
random.seed(CONFIG['seed']); np.random.seed(CONFIG['seed']); torch.manual_seed(CONFIG['seed'])
if torch.cuda.is_available(): torch.cuda.manual_seed_all(CONFIG['seed'])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print('Dispositivo:', DEVICE)

## 2. Dataset

Disponibilize o dataset existente em `data/DeepPCB/PCBData/` no projeto local ou em `MyDrive/vision-transfer-learning/dataset/DeepPCB/PCBData/` no Colab. A pasta deve conter `trainval.txt`, `test.txt` e os grupos referenciados. O notebook não baixa nem duplica o dataset.

In [ ]:
if not PCB.is_dir():
    raise FileNotFoundError(f'Diretório do dataset não encontrado: {PCB}')
for name in ('trainval.txt', 'test.txt'):
    if not (PCB / name).is_file():
        raise FileNotFoundError(f'Split obrigatório não encontrado: {PCB / name}')
print('Estrutura inicial do dataset encontrada em:', PCB)

Os caminhos dos splits usam o nome base da imagem; as imagens reais terminam em `_test.jpg` e `_temp.jpg`. As anotações mantêm coordenadas XYXY absolutas. A leitura rejeita arquivos ausentes e anotações inválidas.

In [ ]:
def read_split(name):
    rows=[]; annotations=[]
    for line_no, line in enumerate((PCB/f'{name}.txt').read_text(encoding='utf-8').splitlines(),1):
        parts=line.split()
        if len(parts)!=2: raise ValueError(f'{name}:{line_no}: esperadas duas colunas')
        if any('\\' in part or ':' in part for part in parts):
            raise ValueError(f'Caminho inválido em {name}:{line_no}')
        image_ref,ann_ref=map(PurePosixPath,parts)
        if any(p.is_absolute() or '..' in p.parts for p in (image_ref,ann_ref)):
            raise ValueError(f'Caminho inválido em {name}:{line_no}')
        base=PCB.joinpath(*image_ref.parts).with_suffix('')
        image=Path(str(base)+'_test.jpg'); template=Path(str(base)+'_temp.jpg'); annotation=PCB.joinpath(*ann_ref.parts)
        if image_ref.stem != ann_ref.stem or image_ref.parts[0] != ann_ref.parts[0]:
            raise ValueError(f'Imagem e anotação incompatíveis: {image_ref}, {ann_ref}')
        for path in (image,template,annotation):
            if not path.is_file(): raise FileNotFoundError(path)
        try:
            with Image.open(image) as im:
                width,height=im.size; im.load()
            with Image.open(template) as im:
                if im.size != (width,height): raise ValueError(f'Dimensões diferentes no par {image_ref.stem}')
                im.load()
        except (OSError, SyntaxError) as exc:
            raise ValueError(f'Imagem inválida no par {image_ref.stem}: {exc}') from exc
        rows.append({'image_id':image_ref.stem,'group_id':image_ref.parts[0], 'image_path':image,
                     'template_path':template,'annotation_path':annotation,'split':name})
        lines=annotation.read_text(encoding='utf-8').splitlines()
        if not lines: raise ValueError(f'Anotação vazia: {annotation}')
        for number,text in enumerate(lines,1):
            try:
                values=[float(v) for v in text.split()]
                if len(values)!=5 or not all(math.isfinite(v) for v in values): raise ValueError('esperados cinco números finitos')
                x1,y1,x2,y2,label=values
                if not label.is_integer() or int(label) not in CLASSES: raise ValueError('classe inválida')
                if not (0<=x1<x2<=width and 0<=y1<y2<=height): raise ValueError('caixa inválida')
            except ValueError as exc: raise ValueError(f'{annotation}:{number}: {exc}') from exc
            annotations.append({'image_id':image_ref.stem,'class_id':int(label),'class_name':CLASSES[int(label)],'x1':x1,'y1':y1,'x2':x2,'y2':y2})
    if not rows: raise ValueError(f'Split vazio: {PCB / (name + '.txt')}')
    return rows,annotations
trainval_rows,trainval_ann=read_split('trainval'); test_rows,test_ann=read_split('test')
summary=pd.DataFrame(trainval_rows+test_rows); ann=pd.DataFrame(trainval_ann+test_ann)
if summary[summary.split=='trainval'].group_id.nunique() < 2: raise ValueError('Treino/validação exige pelo menos dois grupos')
if summary.image_id.duplicated().any(): raise ValueError('IDs repetidos nos splits oficiais')
print('Imagens:',len(summary),'| anotações:',len(ann))
display(ann.class_id.map(CLASSES).value_counts().rename('defeitos').to_frame())

## 3. Divisão e visualização

A validação usa grupos exclusivos, com proporção próxima de 15% e cobertura de classes quando possível. O teste oficial compartilha grupos com o conjunto de desenvolvimento: seus resultados não demonstram generalização para grupos independentes.

In [ ]:
tv=summary[summary.split=='trainval'].copy(); test=summary[summary.split=='test'].copy()
def choose_validation_groups(frame, annotations, seed):
    groups=sorted(frame.group_id.unique()); candidates=[]
    for n in range(1,len(groups)):
        for chosen in combinations(groups,n):
            mask=frame.group_id.isin(chosen)
            left=set(annotations[annotations.image_id.isin(frame.loc[~mask,'image_id'])].class_id)
            right=set(annotations[annotations.image_id.isin(frame.loc[mask,'image_id'])].class_id)
            missing=len(set(CLASSES)-left)+len(set(CLASSES)-right)
            tie=hashlib.sha256((str(seed)+'|'+','.join(chosen)).encode()).hexdigest()
            candidates.append((missing,abs(int(mask.sum())-0.15*len(frame)),tie,chosen))
    return min(candidates)[-1]
validation_groups=choose_validation_groups(tv,ann,CONFIG['seed'])
val=tv[tv.group_id.isin(validation_groups)].copy(); train=tv[~tv.group_id.isin(validation_groups)].copy()
train['split']='train'; val['split']='validation'; split_df=pd.concat([train,val,test],ignore_index=True)
ids={s:set(g.image_id) for s,g in split_df.groupby('split')}; groups={s:set(g.group_id) for s,g in split_df.groupby('split')}
overlap={f'{a}-{b}':sorted(groups[a]&groups[b]) for a,b in [('train','validation'),('train','test'),('validation','test')]}
print('Grupos compartilhados:',overlap); print('Contagens:',split_df.split.value_counts().to_dict())
class_dist=ann.merge(split_df[['image_id','split']],on='image_id').groupby(['split','class_name']).size().unstack(fill_value=0); display(class_dist)

assert len(split_df)==len(summary) and split_df.image_id.is_unique and not overlap['train-validation']

In [ ]:
row=train.iloc[0]; boxes=ann[ann.image_id==row.image_id]
fig,axes=plt.subplots(1,2,figsize=(11,5))
with Image.open(row.image_path) as im: axes[0].imshow(im.convert('RGB'))
with Image.open(row.template_path) as im: axes[1].imshow(im.convert('RGB'))
for box in boxes.itertuples():
    color=plt.cm.tab10(box.class_id-1)
    axes[0].add_patch(Rectangle((box.x1,box.y1),box.x2-box.x1,box.y2-box.y1,fill=False,edgecolor=color,linewidth=1.5))
    axes[0].text(box.x1,max(0,box.y1-3),CLASSES[box.class_id],color='white',fontsize=7,bbox={'facecolor':color,'pad':1})
axes[0].set_title('Defeitos'); axes[1].set_title('Referência')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## 4. PyTorch Dataset e modelo

O modelo recebe apenas a imagem defeituosa, convertida para RGB em `[0,1]`. O experimento A não usa pesos COCO nem ImageNet; B congela o backbone e a FPN; C ajusta todos os parâmetros.

In [ ]:
stable_ids={v:i for i,v in enumerate(sorted(summary.image_id.unique()))}
class PCBDetectionDataset(Dataset):
    def __init__(self, frame, annotations):
        self.frame=frame.reset_index(drop=True).copy()
        self.annotations={k:g for k,g in annotations.groupby('image_id')}
    def __len__(self): return len(self.frame)
    def __getitem__(self, index):
        row=self.frame.iloc[index]
        with Image.open(row.image_path) as im: image=pil_to_tensor(im.convert('RGB')).float()/255.0
        a=self.annotations.get(row.image_id,ann.iloc[:0])
        boxes=torch.as_tensor(a[['x1','y1','x2','y2']].to_numpy(dtype=np.float32,copy=True)).reshape(-1,4)
        labels=torch.as_tensor(a.class_id.to_numpy(dtype=np.int64,copy=True))
        target={'boxes':boxes,'labels':labels,'image_id':torch.tensor(stable_ids[row.image_id],dtype=torch.int64),
                'area':(boxes[:,2]-boxes[:,0])*(boxes[:,3]-boxes[:,1]),'iscrowd':torch.zeros(len(boxes),dtype=torch.int64)}
        return image,target
def collate_fn(batch): return tuple(zip(*batch))
datasets={s:PCBDetectionDataset(split_df[split_df.split==s],ann) for s in ['train','validation','test']}
def make_loader(split, shuffle=False):
    generator=torch.Generator().manual_seed(CONFIG['seed'])
    return DataLoader(datasets[split],batch_size=CONFIG['batch_size'],shuffle=shuffle,num_workers=CONFIG['num_workers'],collate_fn=collate_fn,generator=generator)
loaders={s:make_loader(s,s=='train') for s in datasets}

In [ ]:
NUM_CLASSES=7
WEIGHTS=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
def make_model(experiment):
    assert experiment in ('A','B','C')
    # Ambos None são essenciais para o experimento A não baixar pesos ImageNet.
    model=fasterrcnn_resnet50_fpn_v2(weights=None if experiment=='A' else WEIGHTS, weights_backbone=None)
    in_features=model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor=FastRCNNPredictor(in_features,NUM_CLASSES)
    if experiment=='B':
        for p in model.backbone.parameters(): p.requires_grad=False
    if experiment=='C':
        for p in model.parameters(): p.requires_grad=True
    return model

## 5. Métricas e gráficos

Precisão, recall e F1 são agregados sobre todos os defeitos (micro), com os limiares de confiança e IoU de `CONFIG`. Uma detecção correta precisa corresponder à mesma classe; predições e alvos sem correspondência são FP e FN. A matriz usa associação espacial independente da classe: linhas são classes reais e colunas são predições, incluindo fundo. mAP usa o protocolo COCO; o benchmark DeepPCB original usa IoU 0,33 e não é diretamente comparável.

In [ ]:
def detection_counts(pred,target,score_threshold,iou_threshold):
    keep=pred['scores']>=score_threshold
    boxes,labels,scores=(pred[k][keep].cpu() for k in ('boxes','labels','scores'))
    gt,gtlabels=target['boxes'].cpu(),target['labels'].cpu()
    ious=box_iou(boxes,gt) if len(boxes) and len(gt) else torch.zeros((len(boxes),len(gt)))
    order=torch.argsort(scores,descending=True).tolist(); used=set(); tp=0
    for i in order:
        candidates=[j for j in range(len(gt)) if j not in used and labels[i]==gtlabels[j] and ious[i,j]>=iou_threshold]
        if candidates:
            used.add(max(candidates,key=lambda j:float(ious[i,j]))); tp+=1
    counts=(tp,len(boxes)-tp,len(gt)-tp)
    confusion=np.zeros((7,7),dtype=np.int64); used=set()
    for i in order:
        candidates=[j for j in range(len(gt)) if j not in used and ious[i,j]>=iou_threshold]
        if candidates:
            j=max(candidates,key=lambda j:float(ious[i,j])); used.add(j)
            confusion[int(gtlabels[j]),int(labels[i])]+=1
        else: confusion[0,int(labels[i])]+=1
    for j in range(len(gt)):
        if j not in used: confusion[int(gtlabels[j]),0]+=1
    return counts,confusion

def evaluate(model,loader,device,config):
    metric=MeanAveragePrecision(box_format='xyxy',iou_type='bbox',backend='pycocotools')
    tp=fp=fn=0; confusion=np.zeros((7,7),dtype=np.int64)
    model.eval()
    with torch.no_grad():
        for images,targets in tqdm(loader,desc='Avaliação',leave=False):
            predictions=[{k:v.detach().cpu() for k,v in p.items()} for p in model([x.to(device) for x in images])]
            metric.update(predictions,[{k:t[k].cpu() for k in ('boxes','labels')} for t in targets])
            for prediction,target in zip(predictions,targets):
                (a,b,c),matrix=detection_counts(prediction,target,config['score_threshold'],config['iou_threshold'])
                tp+=a; fp+=b; fn+=c; confusion+=matrix
    coco=metric.compute(); precision=tp/(tp+fp) if tp+fp else 0.; recall=tp/(tp+fn) if tp+fn else 0.
    f1=2*precision*recall/(precision+recall) if precision+recall else 0.
    values={'precision':precision,'recall':recall,'f1':f1,'map_50':float(coco['map_50']),
            'map_50_95':float(coco['map']),'tp':tp,'fp':fp,'fn':fn}
    return values,confusion

def plot_history(history,run_dir):
    frame=pd.DataFrame(history); fig,axes=plt.subplots(1,2,figsize=(12,4))
    for key in ('precision','recall','f1'): axes[0].plot(frame.epoch,frame[key],marker='o',label=key)
    axes[0].set(xlabel='Época',ylabel='Valor',ylim=(0,1.02),title='Validação'); axes[0].legend(); axes[0].grid(alpha=.3)
    axes[1].plot(frame.epoch,frame.train_loss,marker='o',label='Perda de treino')
    axes[1].set(xlabel='Época',ylabel='Perda',title='Treinamento'); axes[1].grid(alpha=.3)
    fig.tight_layout(); fig.savefig(run_dir/'figures'/'training_curves.png',dpi=150); display(fig); plt.close(fig)

def plot_confusion(matrix,run_dir,tag):
    names=['fundo']+list(CLASSES.values()); fig,ax=plt.subplots(figsize=(8,7)); im=ax.imshow(matrix,cmap='Blues')
    ax.set(xticks=range(7),yticks=range(7),xticklabels=names,yticklabels=names,xlabel='Predição',ylabel='Classe real',title=f'Matriz de confusão — {tag}')
    plt.setp(ax.get_xticklabels(),rotation=45,ha='right')
    for i in range(7):
        for j in range(7): ax.text(j,i,str(matrix[i,j]),ha='center',va='center',color='white' if matrix[i,j]>matrix.max()/2 else 'black')
    fig.colorbar(im,ax=ax); fig.tight_layout(); fig.savefig(run_dir/'figures'/f'confusion_{tag}.png',dpi=150); display(fig); plt.close(fig)

## 6. Treinamento e avaliação

Cada rodada cria um diretório exclusivo em `runs/results/` no projeto local ou em `MyDrive/vision-transfer-learning/runs/results/` no Colab, com configuração, divisão, histórico, figuras e melhor checkpoint. As curvas e a matriz de validação correspondem às métricas calculadas, e não são criadas enquanto não houver execução. Para avaliar o teste oficial depois da seleção do experimento, defina `evaluate_test=True` e informe `checkpoint_path`, ou execute o teste ao fim do treinamento selecionado. Caminhos relativos de checkpoint partem da raiz do projeto no ambiente ativo.

In [ ]:
def create_run(config):
    RESULTS.mkdir(parents=True,exist_ok=True)
    while True:
        run_dir=RESULTS/(datetime.now().strftime('%Y%m%d_%H%M%S_%f')+'_'+config['experiment'])
        try: run_dir.mkdir(exist_ok=False); break
        except FileExistsError: continue
    (run_dir/'figures').mkdir(); (run_dir/'checkpoints').mkdir()
    (run_dir/'config.json').write_text(json.dumps(config,indent=2,ensure_ascii=False),encoding='utf-8')
    # Caminhos relativos tornam a divisão portátil entre instalações.
    manifest=split_df[['image_id','group_id','split']].copy(); manifest.to_csv(run_dir/'split.csv',index=False)
    return run_dir

def train_one_epoch(model,loader,optimizer,device):
    model.train()
    if not any(p.requires_grad for p in model.backbone.parameters()): model.backbone.eval()
    total=0.
    for images,targets in tqdm(loader,desc='Treino',leave=False):
        images=[x.to(device) for x in images]; targets=[{k:v.to(device) for k,v in t.items()} for t in targets]
        loss=sum(model(images,targets).values())
        if not torch.isfinite(loss): raise RuntimeError('Perda não finita')
        optimizer.zero_grad(); loss.backward(); optimizer.step(); total+=loss.item()
    return total/len(loader)

def train_experiment(config,run_dir):
    random.seed(config['seed']); np.random.seed(config['seed']); torch.manual_seed(config['seed'])
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(config['seed'])
    model=make_model(config['experiment']).to(DEVICE)
    optimizer=torch.optim.SGD((p for p in model.parameters() if p.requires_grad),lr=config['learning_rate'],momentum=config['momentum'],weight_decay=config['weight_decay'])
    scheduler=torch.optim.lr_scheduler.StepLR(optimizer,step_size=config['lr_step_size'],gamma=config['lr_gamma'])
    history=[]; best=-1.; stale=0; train_loader=make_loader('train',True)
    checkpoint=run_dir/'checkpoints'/'best.pt'
    for epoch in range(config['epochs']):
        loss=train_one_epoch(model,train_loader,optimizer,DEVICE)
        metrics,confusion=evaluate(model,loaders['validation'],DEVICE,config); scheduler.step()
        score=metrics['map_50_95']
        if not math.isfinite(score) or score<0: raise RuntimeError('mAP de validação indisponível')
        history.append({'epoch':epoch+1,'train_loss':loss,**metrics})
        pd.DataFrame(history).to_csv(run_dir/'history.csv',index=False); plot_history(history,run_dir)
        if score>best:
            best=score; stale=0
            torch.save({'model':model.state_dict(),'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
                        'epoch':epoch+1,'config':dict(config),'best_map':best},checkpoint)
            plot_confusion(confusion,run_dir,'validation')
            np.save(run_dir/'confusion_validation.npy',confusion)
        else: stale+=1
        print(history[-1])
        if stale>=config['patience']: print('Early stopping'); break
    return checkpoint

if CONFIG['run_training'] or CONFIG['evaluate_test']:
    if CONFIG['experiment'] not in ('A','B','C'): raise ValueError('Experimento deve ser A, B ou C')
    if CONFIG['epochs']<1 or CONFIG['patience']<1: raise ValueError('epochs e patience devem ser positivos')
    if not all(0<=CONFIG[k]<=1 for k in ('score_threshold','iou_threshold')): raise ValueError('Limiares devem estar entre 0 e 1')
    run_dir=create_run(CONFIG); print('Rodada:',run_dir)
    checkpoint=train_experiment(CONFIG,run_dir) if CONFIG['run_training'] else CONFIG['checkpoint_path']
    if CONFIG['evaluate_test']:
        if not checkpoint: raise ValueError('Informe checkpoint_path para avaliar um modelo existente')
        checkpoint=Path(checkpoint); checkpoint=checkpoint if checkpoint.is_absolute() else ROOT/checkpoint
        # Carregue apenas checkpoints de origem confiável.
        saved=torch.load(checkpoint,map_location='cpu',weights_only=False)
        test_model=make_model(saved['config']['experiment']).to(DEVICE); test_model.load_state_dict(saved['model'])
        test_metrics,test_confusion=evaluate(test_model,loaders['test'],DEVICE,CONFIG)
        (run_dir/'test_metrics.json').write_text(json.dumps(test_metrics,indent=2),encoding='utf-8')
        np.save(run_dir/'confusion_test.npy',test_confusion); plot_confusion(test_confusion,run_dir,'test')
        print('Teste:',test_metrics)
else:
    print('Treinamento e avaliação desativados. Ajuste CONFIG para iniciar uma rodada.')